# Reload graph with attributes

In [8]:
import igraph as ig
import glob, os

indir = "/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/subgraphs"

group_graphs_loaded = {}
for path in glob.glob(os.path.join(indir, "*.graphml")):
    group = os.path.basename(path).replace(".graphml", "")
    group_graphs_loaded[group] = ig.Graph.Read_GraphML(path)

print(f"✅ Reloaded {len(group_graphs_loaded)} subnetworks")


✅ Reloaded 12 subnetworks


In [9]:
def inspect_subgraphs(group_graphs):
    """
    Print a quick summary of loaded subgraphs
    """
    print(f"📊 Found {len(group_graphs)} subnetworks\n")
    
    for name, g in group_graphs.items():
        print(f"🔹 {name}")
        print(f"   Nodes: {g.vcount()}, Edges: {g.ecount()}")
        
        # Show available attributes
        print(f"   Vertex attributes: {g.vs.attributes()}")
        print(f"   Edge attributes:   {g.es.attributes()}")
        
        # Peek at first few vertices
        for v in g.vs[:3]:
            attrs = {attr: v[attr] for attr in g.vs.attributes() if attr in v.attributes()}
            print("     ", attrs)
        
        print("   ...")
        print("-" * 50)


# Example usage:
inspect_subgraphs(group_graphs_loaded)


📊 Found 12 subnetworks

🔹 0_t_0.0000-0.5000
   Nodes: 119, Edges: 128
   Vertex attributes: ['name', 'gene_symbol', 'id']
   Edge attributes:   ['type']
      {'name': 'Q05209', 'gene_symbol': 'PTPN12', 'id': 'n0'}
      {'name': 'Q05397', 'gene_symbol': 'PTK2', 'id': 'n1'}
      {'name': 'Q15311', 'gene_symbol': 'RALBP1', 'id': 'n2'}
   ...
--------------------------------------------------
🔹 3_t_1.0000-1.5000
   Nodes: 169, Edges: 199
   Vertex attributes: ['name', 'gene_symbol', 'id']
   Edge attributes:   ['type']
      {'name': 'P06493', 'gene_symbol': 'CDK1', 'id': 'n0'}
      {'name': 'P31350', 'gene_symbol': 'RRM2', 'id': 'n1'}
      {'name': 'Q05209', 'gene_symbol': 'PTPN12', 'id': 'n2'}
   ...
--------------------------------------------------
🔹 3_t_0.0000-1.0000
   Nodes: 164, Edges: 196
   Vertex attributes: ['name', 'gene_symbol', 'id']
   Edge attributes:   ['type']
      {'name': 'Q05209', 'gene_symbol': 'PTPN12', 'id': 'n0'}
      {'name': 'Q05397', 'gene_symbol': 'PTK2

In [10]:
import os
import pandas as pd
import igraph as ig

def load_full_graph(graphml_path,
                    nodes_csv=None,
                    edges_csv=None,
                    preview_n=5,
                    required_vertex_attrs=("name","gene_symbol","ensembl_gene_id"),
                    required_edge_attrs=("type",)):
    """
    Reload a full graph from GraphML and print a compact summary.
    Optionally load node/edge CSVs (if you saved them) and compare counts/keys.
    """
    assert os.path.exists(graphml_path), f"Missing file: {graphml_path}"
    g = ig.Graph.Read_GraphML(graphml_path)
    print(f"✅ Loaded graphml: {graphml_path}")
    print(f"   Vertices: {g.vcount()}   Edges: {g.ecount()}")
    print(f"   Vertex attrs: {g.vs.attributes()}")
    print(f"   Edge   attrs: {g.es.attributes()}")

    # quick attribute presence check
    missing_v = [a for a in required_vertex_attrs if a not in g.vs.attributes()]
    missing_e = [a for a in required_edge_attrs if a not in g.es.attributes()]
    if missing_v:
        print(f"⚠️ Missing vertex attrs: {missing_v}")
    if missing_e:
        print(f"⚠️ Missing edge attrs: {missing_e}")

    # vertex preview
    print("\n🧬 Vertex preview:")
    vdf = pd.DataFrame({attr: g.vs[attr] for attr in g.vs.attributes()})
    print(vdf.head(preview_n))

    # edge preview (with endpoints resolved)
    print("\n🕸️ Edge preview:")
    edf = pd.DataFrame({attr: g.es[attr] for attr in g.es.attributes()}) if g.es.attributes() else pd.DataFrame()
    src = [e.tuple[0] for e in g.es]
    tgt = [e.tuple[1] for e in g.es]
    edf.insert(0, "source_idx", src)
    edf.insert(1, "target_idx", tgt)
    if "name" in g.vs.attributes():
        idx2name = dict(enumerate(g.vs["name"]))
        edf["source_name"] = [idx2name[i] for i in src]
        edf["target_name"] = [idx2name[i] for i in tgt]
    print(edf.head(preview_n))

    # optional: compare to CSVs you saved
    if nodes_csv and os.path.exists(nodes_csv):
        nodes_tab = pd.read_csv(nodes_csv)
        print(f"\n📄 Loaded node CSV: {nodes_csv}  (rows={len(nodes_tab)})")
        print(nodes_tab.head(min(preview_n, len(nodes_tab))))
    if edges_csv and os.path.exists(edges_csv):
        edges_tab = pd.read_csv(edges_csv)
        print(f"\n📄 Loaded edge CSV: {edges_csv}  (rows={len(edges_tab)})")
        print(edges_tab.head(min(preview_n, len(edges_tab))))

    return g

# --- usage ---
g_full = load_full_graph(
    graphml_path="/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/graph_with_attributes.graphml",
    nodes_csv="/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/graph_vertices.csv",
    edges_csv="/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/graph_edges.csv",
    preview_n=10
)


✅ Loaded graphml: /bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/graph_with_attributes.graphml
   Vertices: 680   Edges: 1403
   Vertex attrs: ['name', 'gene_symbol', 'module', 'uniprot_id', 'ensembl_gene_id', 'id']
   Edge   attrs: ['type']

🧬 Vertex preview:
     name gene_symbol  module uniprot_id  ensembl_gene_id  id
0   CALM3       CALM3     0.0     P0DP24  ENSG00000143933  n0
1   TRPC1       TRPC1     0.0     P48995  ENSG00000144935  n1
2    CAV1        CAV1     1.0     Q03135  ENSG00000105974  n2
3   ITPR2       ITPR2     0.0     Q14571  ENSG00000123104  n3
4   STIM1       STIM1     4.0     Q13586  ENSG00000167323  n4
5    CDK1        CDK1     5.0     P06493  ENSG00000170312  n5
6    RRM2        RRM2     5.0     P31350  ENSG00000171848  n6
7  CDKN1A      CDKN1A     6.0     P38936  ENSG00000124762  n7
8    TP53        TP53     2.0     P04637  ENSG00000141510  n8
9    ATF3        ATF3     4.0     P18847  ENSG00000162772  n9

🕸️ Edge preview:
   source_idx  target

In [11]:
g_full

# Find which modules become suddenly connected

In [2]:
import leidenalg

def assign_modules(G, resolution=1.0):
    part = leidenalg.find_partition(
        G.as_undirected(),
        leidenalg.RBConfigurationVertexPartition,
        resolution_parameter=resolution
    )
    G.vs["module"] = part.membership
    return G

def module_edge_table(G):
    mu = G.vs["module"]
    rows = []
    for e in G.es:
        s, t = e.tuple
        if mu[s] != mu[t]:   # only inter-module edges
            rows.append(tuple(sorted([mu[s], mu[t]])))
    df = pd.Series(rows).value_counts().reset_index()
    df.columns = ["pair","count"]
    return df


# Compare two graphs

In [4]:
def module_bridge_delta(G_prev, G_next, resolution=1.0):
    assign_modules(G_prev, resolution=resolution)
    assign_modules(G_next, resolution=resolution)

    A = module_edge_table(G_prev).set_index("pair")["count"]
    B = module_edge_table(G_next).set_index("pair")["count"]

    all_pairs = A.index.union(B.index)
    A = A.reindex(all_pairs, fill_value=0)
    B = B.reindex(all_pairs, fill_value=0)

    out = pd.DataFrame({
        "w_prev": A, "w_next": B, "delta": B - A
    }).reset_index()
    return out.sort_values("delta", ascending=False)


# Which genes actually drive those new bridges?

In [5]:
def new_cross_edges(G_prev, G_next):
    prev_set = {(G_prev.vs[e.source]["name"], G_prev.vs[e.target]["name"]) for e in G_prev.es}
    added = []
    for e in G_next.es:
        s = G_next.vs[e.source]["name"]
        t = G_next.vs[e.target]["name"]
        if (s, t) not in prev_set:
            added.append((s,t))
    return pd.DataFrame(added, columns=["source","target"])

def cross_module_contributors(G_next, new_edges_df, top_k=20):
    mu = G_next.vs["module"]
    name2idx = {v["name"]: i for i,v in enumerate(G_next.vs)}

    rows = []
    for _, r in new_edges_df.iterrows():
        s, t = r["source"], r["target"]
        if s in name2idx and t in name2idx:
            if mu[name2idx[s]] != mu[name2idx[t]]:
                rows += [s,t]
    return pd.Series(rows).value_counts().head(top_k)


# Bonus: “Participation coefficient” (module-bridging score)

In [ ]:
from collections import Counter

def participation_coefficient(G):
    mu = G.vs["module"]
    names = g.vs["name"]
    P = {}
    for v in range(G.vcount()):
        neigh = G.neighbors(v)
        k = len(neigh)
        if k == 0:
            P[names[v]] = 0
            continue
        counts = Counter(mu[n] for n in neigh)
        P[names[v]] = 1 - sum((c/k)**2 for c in counts.values())
    return pd.Series(P)


# Example workflow

In [6]:
# Suppose you have two condition-specific graphs:
g_prev = ig.Graph.Read_GraphML("graph_condition_prev.graphml")
g_next = ig.Graph.Read_GraphML("graph_condition_next.graphml")

# Which modules connect?
delta = module_bridge_delta(g_prev, g_next)
print(delta.head(10))

# Which genes drive new inter-module links?
added_edges = new_cross_edges(g_prev, g_next)
top_genes = cross_module_contributors(g_next, added_edges)
print(top_genes)

# Participation change
P0 = participation_coefficient(assign_modules(g_prev))
P1 = participation_coefficient(assign_modules(g_next))
dP = (P1 - P0).sort_values(ascending=False)
print(dP.head(10))


FileNotFoundError: [Errno 2] No such file or directory: 'graph_condition_prev.graphml'